# 02 · 메인 eval — **150k 체크포인트 × 5회 반복**

**프로토콜 (확정)**
- 평가 대상 = **150k 체크포인트 하나** (`cf.CKPT_STEP`). best-ckpt 고르기 없음 → 모델 간 공정.
- **반복 5회** (`cf.EVAL_REPEATS`). rep 마다 `--seed = 1000 + 100*rep` → **env 초기상태가 달라짐**.
  → 학습 seed(모델 분산)와 rep(평가 분산)을 **분리**해서 볼 수 있음.
- 모델 = `cf.FINAL_TAGS` 7개. `act_te` 는 **act 체크포인트 재사용 + TE 플래그**(재학습 X).
- 각 run 마다 action(.pt) 기록 → `04_report_jerk` 가 rep 5개를 전부 pool 해서 떨림 계산.

총 run = 7모델 × 4seed × 5rep = **140** (× `N_EP` 에피소드). 이미 끝난 run 은 자동 skip(resume).


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import importlib, common_final as cf
importlib.reload(cf)

TASK  = cf.MAIN_SIM              # 'insertion'
SEEDS = cf.MAIN_SEEDS            # [0,1,2,3]
TAGS  = cf.FINAL_TAGS            # 7모델 (act_te 포함)
REPS  = list(range(cf.EVAL_REPEATS))   # [0,1,2,3,4]
N_EP  = cf.EVAL_N_EP             # rep 1회당 에피소드
NGPU  = 8

print('ckpt :', f'{cf.CKPT_STEP:,}', '| reps:', REPS, '| n_ep/rep:', N_EP)
print('모델 :', TAGS)
print('총 run:', len(TAGS) * len(SEEDS) * len(REPS), f'(에피소드 {len(TAGS)*len(SEEDS)*len(REPS)*N_EP:,})')

## 사전 확인 — 150k 체크포인트가 있는가 (없으면 그 모델/seed 는 eval 불가)

In [ ]:
missing = []
for s in SEEDS:
    row = []
    for t in cf.TRAIN_TAGS:                      # act_te 는 act ckpt 를 씀
        cd = cf.v23.best_ckpt_dir(t, s, TASK, how=cf.CKPT_STEP)
        if cd is None:
            row.append(f'{t}:X'); missing.append((t, s))
        else:
            step = int(cd.name)
            row.append(f'{t}:{step // 1000}k' + ('' if step == cf.CKPT_STEP else ' (!=150k)'))
    print(f'  seed{s}  ' + '  '.join(row))
if missing:
    print('\n⚠ 체크포인트 없음:', missing, '→ 01_train_main 먼저')

## 반복 eval 실행 — 140 run 을 NGPU 청크로
이미 `eval_info.json` 이 있는 run 은 **skip**(중단 후 재실행해도 안전).

In [ ]:
jobs = [(t, s, r) for s in SEEDS for t in TAGS for r in REPS]
todo = [(t, s, r) for (t, s, r) in jobs if cf.rep_sr(t, s, TASK, r) is None]
print(f'전체 {len(jobs)} run / 남은 {len(todo)} run (완료 {len(jobs) - len(todo)})')

for i in range(0, len(todo), NGPU):
    chunk = todo[i:i + NGPU]
    labeled = []
    for g, (t, s, r) in enumerate(chunk):
        try:
            labeled.append((f'{t}/seed{s}/rep{r}',
                            cf.repeat_eval_cmd(t, s, r, task=TASK, gpu_id=g, n_episodes=N_EP)))
        except FileNotFoundError as e:
            print('  skip:', t, s, r, e)
    if not labeled:
        continue
    print('\n===== eval 청크 %d/%d (%d run) =====' % (i // NGPU + 1, -(-len(todo) // NGPU), len(labeled)))
    cf.launch_cmds_live(labeled)
print('\n반복 eval 완료')

## 결과 — 모델별 SR (rep 5회 × seed 4개 = 20 run)
- **mean ± std(20 run)** = 최종 수치 (std 가 곧 평가+학습 랜덤성).
- seed별 평균도 같이 → 모델 분산(seed) vs 평가 분산(rep) 비교.
- pooled Wilson 95% CI = 전체 에피소드(20 × N_EP) 기준.

In [ ]:
import csv
import numpy as np

rows = []
for t in TAGS:
    agg = cf.sr_over_reps(t, task=TASK, seeds=SEEDS, reps=REPS)
    if agg['mean'] is None:
        print(f'{t:<12} (결과 없음)')
        continue
    n_ep_total = agg['n_runs'] * N_EP
    k = int(round(agg['mean'] / 100 * n_ep_total))
    lo, hi = cf.wilson_ci(k, n_ep_total)
    rows.append({
        'tag': t, 'model': cf.FINAL_LABELS.get(t, t),
        'SR_mean': round(agg['mean'], 2), 'SR_std': round(agg['std'], 2),
        'n_run': agg['n_runs'], 'n_episodes': n_ep_total,
        'CI_lo': round(lo * 100, 1), 'CI_hi': round(hi * 100, 1),
        'per_seed': {s: round(v, 1) for s, v in agg['per_seed'].items()},
    })

out = cf.OUTPUT_BASE / 'main_report'
out.mkdir(parents=True, exist_ok=True)
with open(out / 'sr_150k_reps.csv', 'w', newline='', encoding='utf-8') as fh:
    w = csv.DictWriter(fh, fieldnames=list(rows[0]))
    w.writeheader()
    w.writerows(rows)

hdr = f"{'MODEL':<34}{'SR (mean±std)':>18}{'95% CI':>16}{'runs':>6}   per-seed"
print(hdr)
print('-' * (len(hdr) + 10))
for r in rows:
    sr = f"{r['SR_mean']:.1f} ± {r['SR_std']:.1f}"
    ci = f"[{r['CI_lo']:.1f}, {r['CI_hi']:.1f}]"
    print(f"{r['model']:<34}{sr:>18}{ci:>16}{r['n_run']:>6}   {r['per_seed']}")
print('\n저장:', out / 'sr_150k_reps.csv')

## 그림 — 모델별 SR (rep 산포 표시)

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
                     'font.size': 13, 'axes.grid': True, 'grid.alpha': 0.3,
                     'axes.spines.top': False, 'axes.spines.right': False})

fig, ax = plt.subplots(figsize=(11, 5))
for i, t in enumerate([r['tag'] for r in rows]):
    agg = cf.sr_over_reps(t, task=TASK, seeds=SEEDS, reps=REPS)
    c = cf.COLOR.get(t, '#333')
    ax.bar(i, agg['mean'], yerr=agg['std'], color=c, alpha=0.85, capsize=5, width=0.62)
    ax.scatter([i] * len(agg['all']), agg['all'], s=12, color='k', alpha=0.35, zorder=3)  # 개별 run
ax.set_xticks(range(len(rows)))
ax.set_xticklabels([cf.FINAL_LABELS.get(r['tag'], r['tag']) for r in rows], rotation=20, ha='right')
ax.set_ylabel('Success rate (%)')
ax.set_title(f'{TASK} @ {cf.CKPT_STEP // 1000}k — {len(REPS)} reps x {len(SEEDS)} seeds (점 = 개별 run)',
             fontweight='bold')
fig.savefig(out / 'sr_150k_reps.png')
fig.savefig(out / 'sr_150k_reps.pdf')
plt.show()
print('저장:', out / 'sr_150k_reps.png')